In [9]:
%run ./functions.ipynb
random.seed(42)

# Exact sampler

In [11]:
# N = 100
class Options:
    pass

N = 100  # Run for 100, 1000 and 10000
d = 10
alpha = np.exp(-1)

if N == 100:
    num_repeats = 10000
elif N == 1000:
    num_repeats = 1000
elif N == 10000:
    num_repeats = 100

# Initialize arrays with NaNs
log_evidence_ans_smc = np.full(num_repeats, np.nan)
count_ans_smc = np.full(num_repeats, np.nan)
time_ans_smc = np.full(num_repeats, np.nan)

log_evidence_ns_smc = np.full(num_repeats, np.nan)
count_ns_smc = np.full(num_repeats, np.nan)
time_ns_smc = np.full(num_repeats, np.nan)
error_flag_ns_smc = np.full(num_repeats, np.nan)

log_evidence_ns = np.full(num_repeats, np.nan)
count_ns = np.full(num_repeats, np.nan)
time_ns = np.full(num_repeats, np.nan)

log_evidence_ns_star = np.full(num_repeats, np.nan)
count_ns_star = np.full(num_repeats, np.nan)
time_ns_star = np.full(num_repeats, np.nan)

for i in range(num_repeats):
    
    # %%%%%%%%%%%%%%%%%%%% PERFORMING ADAPTIVE NS-SMC %%%%%%%%%%%%%%%%%%%%%%%
    options = Options()
    options.N = N
    options.d = d
    options.alpha = alpha
    
    np.random.seed(i)
    
    t_start = time.time()
    log_evidence, count_loglike, levels, dists = ANS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ans_smc[i] = log_evidence
    count_ans_smc[i] = count_loglike
    time_ans_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED NS-SMC %%%%%%%%%%%%%%%%%%%%%%
    options.levels = levels
    options.dists = dists

    np.random.seed(i + num_repeats)
    
    t_start = time.time()
    log_evidence, count_loglike, error_flag = NS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ns_smc[i] = log_evidence
    count_ns_smc[i] = count_loglike + count_ans_smc[i]
    time_ns_smc[i] = t_elapsed + time_ans_smc[i]
    error_flag_ns_smc[i] = error_flag

    # %%%%%%%%%%%%%%%%% PERFORMING NS (WITH MAX EVALS) %%%%%%%%%%%%%%%%%%%%%%
    options_ns = Options()
    options_ns.N = N
    options_ns.d = d
    options_ns.desired_count = count_ns_smc[i]

    np.random.seed(i)
    
    t_start = time.time()
    log_evidence, log_evidence_star_val, count_loglike = NS_bespoke_exact(loglike_fn, simprior_fn, options_ns, False)
    t_elapsed = time.time() - t_start

    log_evidence_ns[i] = log_evidence
    count_ns[i] = count_loglike
    time_ns[i] = t_elapsed
    
    log_evidence_ns_star[i] = log_evidence_star_val
    count_ns_star[i] = count_loglike
    time_ns_star[i] = t_elapsed

    if (i + 1) % (num_repeats // 100) == 0:
        print(f'Iteration {i + 1}.')

Iteration 100.
Iteration 200.
Iteration 300.
Iteration 400.
Iteration 500.
Iteration 600.
Iteration 700.
Iteration 800.
Iteration 900.
Iteration 1000.
Iteration 1100.
Iteration 1200.
Iteration 1300.
Iteration 1400.
Iteration 1500.
Iteration 1600.
Iteration 1700.
Iteration 1800.
Iteration 1900.
Iteration 2000.
Iteration 2100.
Iteration 2200.
Iteration 2300.
Iteration 2400.
Iteration 2500.
Iteration 2600.
Iteration 2700.
Iteration 2800.
Iteration 2900.
Iteration 3000.
Iteration 3100.
Iteration 3200.
Iteration 3300.
Iteration 3400.
Iteration 3500.
Iteration 3600.
Iteration 3700.
Iteration 3800.
Iteration 3900.
Iteration 4000.
Iteration 4100.
Iteration 4200.
Iteration 4300.
Iteration 4400.
Iteration 4500.
Iteration 4600.
Iteration 4700.
Iteration 4800.
Iteration 4900.
Iteration 5000.
Iteration 5100.
Iteration 5200.
Iteration 5300.
Iteration 5400.
Iteration 5500.
Iteration 5600.
Iteration 5700.
Iteration 5800.
Iteration 5900.
Iteration 6000.
Iteration 6100.
Iteration 6200.
Iteration 6300.
I

In [12]:
# 1. Analytical ground truth for the 10D Spike-and-Slab on the unit ball
# Since prior is uniform on unit d-ball (volume V_d = pi^(d/2) / Gamma(d/2 + 1)),
# the integral of each Gaussian component inside radius 1 has an exact Incomplete Gamma form:
d = 10
u, v = 0.1, 0.01
a1, a2 = 0.1, 0.9
# Note: in the paper, the prior density is normalized or unnormalized depending on logprior_fn.
# Since logprior_fn returns 0 inside the unit ball (unnormalized density = 1), 
# the integral of a Gaussian N(0, sig^2 I) inside radius R=1 is gammainc(d/2, 1/(2*sig^2)):
Z_true = a1 * gammainc(d / 2.0, 1.0 / (2.0 * u**2)) + a2 * gammainc(d / 2.0, 1.0 / (2.0 * v**2))
# Note: If comparing directly to the paper's 0.3921, check whether the paper's 
# loglike_fn or prior normalization constant scales this by a constant factor, 
# or set Z_true = 0.3921 directly as reported in Table 2's caption.
Z_true_paper = 0.39210 

methods = [
    ("NS", log_evidence_ns, count_ns),
    ("NS*", log_evidence_ns_star, count_ns_star),
    ("ANS-SMC", log_evidence_ans_smc, count_ans_smc),
    ("NS-SMC", log_evidence_ns_smc, count_ns_smc),
]

print(f"--- Table 2 Results (Exact Sampler, N = {N}, {num_repeats} repeats) ---")
print(f"{'Method':<10} | {'Z^N (SE)':<18} | {'Evals':<10} | {'MSE':<12} | {'p-val (<0.05/30?)'}")
print("-" * 75)

mse_list = []
results_summary = []

for name, log_ev_arr, count_arr in methods:
    # Convert from log-evidence to raw evidence Z
    Z_est = np.exp(log_ev_arr)
    
    mean_Z = np.mean(Z_est)
    se_Z = np.std(Z_est, ddof=1) / np.sqrt(len(Z_est))
    mean_evals = np.mean(count_arr)
    
    # MSE and t-test against ground truth
    mse = np.mean((Z_est - Z_true_paper)**2)
    t_stat, p_val = stats.ttest_1samp(Z_est, Z_true_paper)
    is_biased = p_val < (0.05 / 30.0)
    
    mse_list.append(mse)
    results_summary.append((name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased))

best_mse_idx = np.argmin(mse_list)

for idx, (name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased) in enumerate(results_summary):
    flag_italic = "ITALIC (Biased)" if is_biased else ""
    flag_bold = "BOLD (Best MSE)" if idx == best_mse_idx else ""
    flags = " ".join(filter(None, [flag_bold, flag_italic]))
    
    print(f"{name:<10} | {mean_Z:.4f} ({se_Z:.4f})   | {mean_evals:.1e}   | {mse:.6e} | {p_val:.4f} {flags}")

--- Table 2 Results (Exact Sampler, N = 100, 10000 repeats) ---
Method     | Z^N (SE)           | Evals      | MSE          | p-val (<0.05/30?)
---------------------------------------------------------------------------
NS         | 0.4555 (0.0027)   | 1.0e+04   | 7.432387e-02 | 0.0000 ITALIC (Biased)
NS*        | 0.3886 (0.0023)   | 1.0e+04   | 5.100228e-02 | 0.1193 BOLD (Best MSE)
ANS-SMC    | 0.3933 (0.0032)   | 5.1e+03   | 1.000693e-01 | 0.6953 
NS-SMC     | 0.3950 (0.0032)   | 1.0e+04   | 1.036612e-01 | 0.3691 


In [2]:
# N = 1000
class Options:
    pass

N = 1000  # Run for 100, 1000 and 10000
d = 10
alpha = np.exp(-1)

if N == 100:
    num_repeats = 10000
elif N == 1000:
    num_repeats = 1000
elif N == 10000:
    num_repeats = 100

# Initialize arrays with NaNs
log_evidence_ans_smc = np.full(num_repeats, np.nan)
count_ans_smc = np.full(num_repeats, np.nan)
time_ans_smc = np.full(num_repeats, np.nan)

log_evidence_ns_smc = np.full(num_repeats, np.nan)
count_ns_smc = np.full(num_repeats, np.nan)
time_ns_smc = np.full(num_repeats, np.nan)
error_flag_ns_smc = np.full(num_repeats, np.nan)

log_evidence_ns = np.full(num_repeats, np.nan)
count_ns = np.full(num_repeats, np.nan)
time_ns = np.full(num_repeats, np.nan)

log_evidence_ns_star = np.full(num_repeats, np.nan)
count_ns_star = np.full(num_repeats, np.nan)
time_ns_star = np.full(num_repeats, np.nan)

for i in range(num_repeats):
    
    # %%%%%%%%%%%%%%%%%%%% PERFORMING ADAPTIVE NS-SMC %%%%%%%%%%%%%%%%%%%%%%%
    options = Options()
    options.N = N
    options.d = d
    options.alpha = alpha
    
    np.random.seed(i)
    
    t_start = time.time()
    log_evidence, count_loglike, levels, dists = ANS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ans_smc[i] = log_evidence
    count_ans_smc[i] = count_loglike
    time_ans_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED NS-SMC %%%%%%%%%%%%%%%%%%%%%%
    options.levels = levels
    options.dists = dists

    np.random.seed(i + num_repeats)
    
    t_start = time.time()
    log_evidence, count_loglike, error_flag = NS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ns_smc[i] = log_evidence
    count_ns_smc[i] = count_loglike + count_ans_smc[i]
    time_ns_smc[i] = t_elapsed + time_ans_smc[i]
    error_flag_ns_smc[i] = error_flag

    # %%%%%%%%%%%%%%%%% PERFORMING NS (WITH MAX EVALS) %%%%%%%%%%%%%%%%%%%%%%
    options_ns = Options()
    options_ns.N = N
    options_ns.d = d
    options_ns.desired_count = count_ns_smc[i]

    np.random.seed(i)
    
    t_start = time.time()
    log_evidence, log_evidence_star_val, count_loglike = NS_bespoke_exact(loglike_fn, simprior_fn, options_ns, False)
    t_elapsed = time.time() - t_start

    log_evidence_ns[i] = log_evidence
    count_ns[i] = count_loglike
    time_ns[i] = t_elapsed
    
    log_evidence_ns_star[i] = log_evidence_star_val
    count_ns_star[i] = count_loglike
    time_ns_star[i] = t_elapsed

    if (i + 1) % (num_repeats // 100) == 0:
        print(f'Iteration {i + 1}.')

Iteration 10.
Iteration 20.
Iteration 30.
Iteration 40.
Iteration 50.
Iteration 60.
Iteration 70.
Iteration 80.
Iteration 90.
Iteration 100.
Iteration 110.
Iteration 120.
Iteration 130.
Iteration 140.
Iteration 150.
Iteration 160.
Iteration 170.
Iteration 180.
Iteration 190.
Iteration 200.
Iteration 210.
Iteration 220.
Iteration 230.
Iteration 240.
Iteration 250.
Iteration 260.
Iteration 270.
Iteration 280.
Iteration 290.
Iteration 300.
Iteration 310.
Iteration 320.
Iteration 330.
Iteration 340.
Iteration 350.
Iteration 360.
Iteration 370.
Iteration 380.
Iteration 390.
Iteration 400.
Iteration 410.
Iteration 420.
Iteration 430.
Iteration 440.
Iteration 450.
Iteration 460.
Iteration 470.
Iteration 480.
Iteration 490.
Iteration 500.
Iteration 510.
Iteration 520.
Iteration 530.
Iteration 540.
Iteration 550.
Iteration 560.
Iteration 570.
Iteration 580.
Iteration 590.
Iteration 600.
Iteration 610.
Iteration 620.
Iteration 630.
Iteration 640.
Iteration 650.
Iteration 660.
Iteration 670.
Iter

In [10]:
d = 10
u, v = 0.1, 0.01
a1, a2 = 0.1, 0.9
Z_true = a1 * gammainc(d / 2.0, 1.0 / (2.0 * u**2)) + a2 * gammainc(d / 2.0, 1.0 / (2.0 * v**2))
Z_true_paper = 0.39210 

methods = [
    ("NS", log_evidence_ns, count_ns),
    ("NS*", log_evidence_ns_star, count_ns_star),
    ("ANS-SMC", log_evidence_ans_smc, count_ans_smc),
    ("NS-SMC", log_evidence_ns_smc, count_ns_smc),
]

print(f"--- Table 2 Results (Exact Sampler, N = {N}, {num_repeats} repeats) ---")
print(f"{'Method':<10} | {'Z^N (SE)':<18} | {'Evals':<10} | {'MSE':<12} | {'p-val (<0.05/30?)'}")
print("-" * 75)

mse_list = []
results_summary = []

for name, log_ev_arr, count_arr in methods:
    # Convert from log-evidence to raw evidence Z
    Z_est = np.exp(log_ev_arr)
    
    mean_Z = np.mean(Z_est)
    se_Z = np.std(Z_est, ddof=1) / np.sqrt(len(Z_est))
    mean_evals = np.mean(count_arr)
    
    # MSE and t-test against ground truth
    mse = np.mean((Z_est - Z_true_paper)**2)
    t_stat, p_val = stats.ttest_1samp(Z_est, Z_true_paper)
    is_biased = p_val < (0.05 / 30.0)
    
    mse_list.append(mse)
    results_summary.append((name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased))

best_mse_idx = np.argmin(mse_list)

for idx, (name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased) in enumerate(results_summary):
    flag_italic = "ITALIC (Biased)" if is_biased else ""
    flag_bold = "BOLD (Best MSE)" if idx == best_mse_idx else ""
    flags = " ".join(filter(None, [flag_bold, flag_italic]))
    
    print(f"{name:<10} | {mean_Z:.4f} ({se_Z:.4f})   | {mean_evals:.1e}   | {mse:.6e} | {p_val:.4f} {flags}")

--- Table 2 Results (Exact Sampler, N = 1000, 1000 repeats) ---
Method     | Z^N (SE)           | Evals      | MSE          | p-val (<0.05/30?)
---------------------------------------------------------------------------
NS         | 0.3990 (0.0022)   | 1.0e+05   | 4.731572e-03 | 0.0014 ITALIC (Biased)
NS*        | 0.3928 (0.0021)   | 1.0e+05   | 4.534988e-03 | 0.7579 BOLD (Best MSE)
ANS-SMC    | 0.3915 (0.0028)   | 5.0e+04   | 7.913307e-03 | 0.8220 
NS-SMC     | 0.3924 (0.0028)   | 1.0e+05   | 7.798666e-03 | 0.9032 


In [13]:
# N = 10000
class Options:
    pass

N = 10000  # Run for 100, 1000 and 10000
d = 10
alpha = np.exp(-1)

if N == 100:
    num_repeats = 10000
elif N == 1000:
    num_repeats = 1000
elif N == 10000:
    num_repeats = 100

# Initialize arrays with NaNs
log_evidence_ans_smc = np.full(num_repeats, np.nan)
count_ans_smc = np.full(num_repeats, np.nan)
time_ans_smc = np.full(num_repeats, np.nan)

log_evidence_ns_smc = np.full(num_repeats, np.nan)
count_ns_smc = np.full(num_repeats, np.nan)
time_ns_smc = np.full(num_repeats, np.nan)
error_flag_ns_smc = np.full(num_repeats, np.nan)

log_evidence_ns = np.full(num_repeats, np.nan)
count_ns = np.full(num_repeats, np.nan)
time_ns = np.full(num_repeats, np.nan)

log_evidence_ns_star = np.full(num_repeats, np.nan)
count_ns_star = np.full(num_repeats, np.nan)
time_ns_star = np.full(num_repeats, np.nan)

for i in range(num_repeats):
    
    # %%%%%%%%%%%%%%%%%%%% PERFORMING ADAPTIVE NS-SMC %%%%%%%%%%%%%%%%%%%%%%%
    options = Options()
    options.N = N
    options.d = d
    options.alpha = alpha
    
    np.random.seed(i)
    
    t_start = time.time()
    log_evidence, count_loglike, levels, dists = ANS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ans_smc[i] = log_evidence
    count_ans_smc[i] = count_loglike
    time_ans_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED NS-SMC %%%%%%%%%%%%%%%%%%%%%%
    options.levels = levels
    options.dists = dists

    np.random.seed(i + num_repeats)
    
    t_start = time.time()
    log_evidence, count_loglike, error_flag = NS_SMC_bespoke_exact(loglike_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ns_smc[i] = log_evidence
    count_ns_smc[i] = count_loglike + count_ans_smc[i]
    time_ns_smc[i] = t_elapsed + time_ans_smc[i]
    error_flag_ns_smc[i] = error_flag

    # %%%%%%%%%%%%%%%%% PERFORMING NS (WITH MAX EVALS) %%%%%%%%%%%%%%%%%%%%%%
    options_ns = Options()
    options_ns.N = N
    options_ns.d = d
    options_ns.desired_count = count_ns_smc[i]

    np.random.seed(i)
    
    t_start = time.time()
    log_evidence, log_evidence_star_val, count_loglike = NS_bespoke_exact(loglike_fn, simprior_fn, options_ns, False)
    t_elapsed = time.time() - t_start

    log_evidence_ns[i] = log_evidence
    count_ns[i] = count_loglike
    time_ns[i] = t_elapsed
    
    log_evidence_ns_star[i] = log_evidence_star_val
    count_ns_star[i] = count_loglike
    time_ns_star[i] = t_elapsed

    if (i + 1) % (num_repeats // 100) == 0:
        print(f'Iteration {i + 1}.')

Iteration 1.
Iteration 2.
Iteration 3.
Iteration 4.
Iteration 5.
Iteration 6.
Iteration 7.
Iteration 8.
Iteration 9.
Iteration 10.
Iteration 11.
Iteration 12.
Iteration 13.
Iteration 14.
Iteration 15.
Iteration 16.
Iteration 17.
Iteration 18.
Iteration 19.
Iteration 20.
Iteration 21.
Iteration 22.
Iteration 23.
Iteration 24.
Iteration 25.
Iteration 26.
Iteration 27.
Iteration 28.
Iteration 29.
Iteration 30.
Iteration 31.
Iteration 32.
Iteration 33.
Iteration 34.
Iteration 35.
Iteration 36.
Iteration 37.
Iteration 38.
Iteration 39.
Iteration 40.
Iteration 41.
Iteration 42.
Iteration 43.
Iteration 44.
Iteration 45.
Iteration 46.
Iteration 47.
Iteration 48.
Iteration 49.
Iteration 50.
Iteration 51.
Iteration 52.
Iteration 53.
Iteration 54.
Iteration 55.
Iteration 56.
Iteration 57.
Iteration 58.
Iteration 59.
Iteration 60.
Iteration 61.
Iteration 62.
Iteration 63.
Iteration 64.
Iteration 65.
Iteration 66.
Iteration 67.
Iteration 68.
Iteration 69.
Iteration 70.
Iteration 71.
Iteration 72.
I

In [14]:
d = 10
u, v = 0.1, 0.01
a1, a2 = 0.1, 0.9
Z_true = a1 * gammainc(d / 2.0, 1.0 / (2.0 * u**2)) + a2 * gammainc(d / 2.0, 1.0 / (2.0 * v**2))
Z_true_paper = 0.39210 

methods = [
    ("NS", log_evidence_ns, count_ns),
    ("NS*", log_evidence_ns_star, count_ns_star),
    ("ANS-SMC", log_evidence_ans_smc, count_ans_smc),
    ("NS-SMC", log_evidence_ns_smc, count_ns_smc),
]

print(f"--- Table 2 Results (Exact Sampler, N = {N}, {num_repeats} repeats) ---")
print(f"{'Method':<10} | {'Z^N (SE)':<18} | {'Evals':<10} | {'MSE':<12} | {'p-val (<0.05/30?)'}")
print("-" * 75)

mse_list = []
results_summary = []

for name, log_ev_arr, count_arr in methods:
    # Convert from log-evidence to raw evidence Z
    Z_est = np.exp(log_ev_arr)
    
    mean_Z = np.mean(Z_est)
    se_Z = np.std(Z_est, ddof=1) / np.sqrt(len(Z_est))
    mean_evals = np.mean(count_arr)
    
    # MSE and t-test against ground truth
    mse = np.mean((Z_est - Z_true_paper)**2)
    t_stat, p_val = stats.ttest_1samp(Z_est, Z_true_paper)
    is_biased = p_val < (0.05 / 30.0)
    
    mse_list.append(mse)
    results_summary.append((name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased))

best_mse_idx = np.argmin(mse_list)

for idx, (name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased) in enumerate(results_summary):
    flag_italic = "ITALIC (Biased)" if is_biased else ""
    flag_bold = "BOLD (Best MSE)" if idx == best_mse_idx else ""
    flags = " ".join(filter(None, [flag_bold, flag_italic]))
    
    print(f"{name:<10} | {mean_Z:.4f} ({se_Z:.4f})   | {mean_evals:.1e}   | {mse:.6e} | {p_val:.4f} {flags}")

--- Table 2 Results (Exact Sampler, N = 10000, 100 repeats) ---
Method     | Z^N (SE)           | Evals      | MSE          | p-val (<0.05/30?)
---------------------------------------------------------------------------
NS         | 0.3961 (0.0023)   | 1.0e+06   | 5.188505e-04 | 0.0753 
NS*        | 0.3955 (0.0022)   | 1.0e+06   | 5.125368e-04 | 0.1315 BOLD (Best MSE)
ANS-SMC    | 0.3966 (0.0026)   | 5.0e+05   | 7.024733e-04 | 0.0917 
NS-SMC     | 0.3914 (0.0026)   | 1.0e+06   | 6.475705e-04 | 0.7821 


# MCMC Sampler

In [19]:
# N = 100
class Options:
    pass

N = 100  # Run for 100, 1000, and 10000
d = 10
alpha = np.exp(-1)
repeats = 10

if N == 100:
    num_repeats = 10000
elif N == 1000:
    num_repeats = 1000
elif N == 10000:
    num_repeats = 100

# Initialize result arrays with NaNs
log_evidence_ata_smc = np.full(num_repeats, np.nan)
count_ata_smc = np.full(num_repeats, np.nan)
time_ata_smc = np.full(num_repeats, np.nan)

log_evidence_ta_smc = np.full(num_repeats, np.nan)
count_ta_smc = np.full(num_repeats, np.nan)
time_ta_smc = np.full(num_repeats, np.nan)

log_evidence_ans_smc = np.full(num_repeats, np.nan)
count_ans_smc = np.full(num_repeats, np.nan)
time_ans_smc = np.full(num_repeats, np.nan)

log_evidence_ns_smc = np.full(num_repeats, np.nan)
count_ns_smc = np.full(num_repeats, np.nan)
time_ns_smc = np.full(num_repeats, np.nan)
error_flag_ns_smc = np.full(num_repeats, np.nan)

log_evidence_ns = np.full(num_repeats, np.nan)
count_ns = np.full(num_repeats, np.nan)
time_ns = np.full(num_repeats, np.nan)

log_evidence_ns_star = np.full(num_repeats, np.nan)
count_ns_star = np.full(num_repeats, np.nan)
time_ns_star = np.full(num_repeats, np.nan)

for i in range(num_repeats):
    
    # %%%%%%%%%%%%%%%%%%%% PERFORMING ATA-SMC %%%%%%%%%%%%%%%%%%%%%%% 
    options = Options()
    options.N = N
    options.d = d
    options.alpha = 0.999
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    # log_unnormalized_w, normalized_w between time t and t+1
    log_evidence, count_loglike, gammavar, ess, log_unnormalized_w, normalized_w, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, log_evidence_t, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = ATA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ata_smc[i] = log_evidence
    count_ata_smc[i] = count_loglike
    time_ata_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED TA-SMC %%%%%%%%%%%%%%%%%%%%%%%
    options.N = N
    options.d = d
    options.gammavar = gammavar
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i + num_repeats)
    t_start = time.time()
    log_evidence, count_loglike = TA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ta_smc[i] = log_evidence
    count_ta_smc[i] = count_loglike + count_ata_smc[i]
    time_ta_smc[i] = t_elapsed + time_ata_smc[i]

    # %%%%%%%%%%%%%%%%%%%%%% PERFORMING ANS-SMC %%%%%%%%%%%%%%%%%%%%% 
    options = Options()
    options.N = N
    options.d = d
    options.alpha = alpha
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    # Log unnormalized and normalized weights for the eliminated particles at each level t
    llog_evidence, count_loglike, levels, log_unnormalized_w, log_normalized_w, log_evidence_t, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = ANS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ans_smc[i] = log_evidence
    count_ans_smc[i] = count_loglike
    time_ans_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED NS-SMC %%%%%%%%%%%%%%%%%%%%%% -1
    options.N = N
    options.d = d
    options.levels = levels # levels is returned from ANS-SMC
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i + num_repeats)
    t_start = time.time()
    # Log unnormalized and normalized weights for the eliminated particles at each level t
    log_evidence, count_loglike, error_flag, log_unnormalized_w, log_normalized_w, log_evidence_t, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = NS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ns_smc[i] = log_evidence
    count_ns_smc[i] = count_loglike + count_ans_smc[i]
    time_ns_smc[i] = t_elapsed + time_ans_smc[i]
    error_flag_ns_smc[i] = error_flag

    # %%%%%%%%%%%%%%%%% PERFORMING NS (WITH MAX EVALS) %%%%%%%%%%%%%%%%%%%%%%
    options_ns = Options()
    options_ns.N = N
    options_ns.d = d
    options_ns.desired_count = count_ns_smc[i]
    options_ns.R = repeats
    options_ns.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    log_evidence, log_evidence_star_val, count_loglike = NS_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options_ns, False)
    t_elapsed = time.time() - t_start

    log_evidence_ns[i] = log_evidence
    count_ns[i] = count_loglike
    time_ns[i] = t_elapsed
    
    log_evidence_ns_star[i] = log_evidence_star_val
    count_ns_star[i] = count_loglike
    time_ns_star[i] = t_elapsed

    # Print progress update -1
    if (i + 1) % max(1, num_repeats // 100) == 0:
        print(f"Iteration {i + 1}.")

Iteration 100.
Iteration 200.
Iteration 300.
Iteration 400.
Iteration 500.
Iteration 600.
Iteration 700.
Iteration 800.
Iteration 900.
Iteration 1000.
Iteration 1100.
Iteration 1200.
Iteration 1300.
Iteration 1400.
Iteration 1500.
Iteration 1600.
Iteration 1700.
Iteration 1800.
Iteration 1900.
Iteration 2000.
Iteration 2100.
Iteration 2200.
Iteration 2300.
Iteration 2400.
Iteration 2500.
Iteration 2600.
Iteration 2700.
Iteration 2800.
Iteration 2900.
Iteration 3000.
Iteration 3100.
Iteration 3200.
Iteration 3300.
Iteration 3400.
Iteration 3500.
Iteration 3600.
Iteration 3700.
Iteration 3800.
Iteration 3900.
Iteration 4000.
Iteration 4100.
Iteration 4200.
Iteration 4300.
Iteration 4400.
Iteration 4500.
Iteration 4600.
Iteration 4700.
Iteration 4800.
Iteration 4900.
Iteration 5000.
Iteration 5100.
Iteration 5200.
Iteration 5300.
Iteration 5400.
Iteration 5500.
Iteration 5600.
Iteration 5700.
Iteration 5800.
Iteration 5900.
Iteration 6000.
Iteration 6100.
Iteration 6200.
Iteration 6300.
I

In [24]:
d = 10
u, v = 0.1, 0.01
a1, a2 = 0.1, 0.9
Z_true = a1 * gammainc(d / 2.0, 1.0 / (2.0 * u**2)) + a2 * gammainc(d / 2.0, 1.0 / (2.0 * v**2))
Z_true_paper = 0.39210 

methods = [
    ("NS", log_evidence_ns, count_ns),
    ("NS*", log_evidence_ns_star, count_ns_star),
    ("ANS-SMC", log_evidence_ans_smc, count_ans_smc),
    ("NS-SMC", log_evidence_ns_smc, count_ns_smc),
    ("ATA-SMC", log_evidence_ata_smc, count_ata_smc),
    ("TA-SMC", log_evidence_ta_smc, count_ta_smc),
]

print(f"--- Table 2 Results (MCMC Sampler, N = {N}, {num_repeats} repeats) ---")
print(f"{'Method':<10} | {'Z^N (SE)':<18} | {'Evals':<10} | {'MSE':<12} | {'p-val (<0.05/30?)'}")
print("-" * 75)

mse_list = []
results_summary = []

for name, log_ev_arr, count_arr in methods:
    # Convert from log-evidence to raw evidence Z
    Z_est = np.exp(log_ev_arr)
    
    mean_Z = np.mean(Z_est)
    se_Z = np.std(Z_est, ddof=1) / np.sqrt(len(Z_est))
    mean_evals = np.mean(count_arr)
    
    # MSE and t-test against ground truth
    mse = np.mean((Z_est - Z_true_paper)**2)
    t_stat, p_val = stats.ttest_1samp(Z_est, Z_true_paper)
    is_biased = p_val < (0.05 / 30.0)
    
    mse_list.append(mse)
    results_summary.append((name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased))

best_mse_idx = np.argmin(mse_list)

for idx, (name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased) in enumerate(results_summary):
    flag_italic = "ITALIC (Biased)" if is_biased else ""
    flag_bold = "BOLD (Best MSE)" if idx == best_mse_idx else ""
    flags = " ".join(filter(None, [flag_bold, flag_italic]))
    
    print(f"{name:<10} | {mean_Z:.4f} ({se_Z:.4f})   | {mean_evals:.1e}   | {mse:.6e} | {p_val:.4f} {flags}")

--- Table 2 Results (MCMC Sampler, N = 10000, 100 repeats) ---
Method     | Z^N (SE)           | Evals      | MSE          | p-val (<0.05/30?)
---------------------------------------------------------------------------
NS         | 0.3978 (0.0057)   | 9.8e+06   | 3.253795e-03 | 0.3240 
NS*        | 0.3971 (0.0057)   | 9.8e+06   | 3.237035e-03 | 0.3798 
ANS-SMC    | 3.8224 (3.7734)   | 4.9e+06   | 1.421374e+03 | 0.3655 
NS-SMC     | 0.3958 (0.0048)   | 9.8e+06   | 2.307767e-03 | 0.4397 BOLD (Best MSE)
ATA-SMC    | 0.8258 (0.5258)   | 1.8e+07   | 2.756074e+01 | 0.4115 
TA-SMC     | 3.8224 (3.7734)   | 3.6e+07   | 1.421374e+03 | 0.3655 


In [21]:
# N = 1000
class Options:
    pass

N = 1000  # Run for 100, 1000, and 10000
d = 10
alpha = np.exp(-1)
repeats = 10

if N == 100:
    num_repeats = 10000
elif N == 1000:
    num_repeats = 1000
elif N == 10000:
    num_repeats = 100

# Initialize result arrays with NaNs
log_evidence_ata_smc = np.full(num_repeats, np.nan)
count_ata_smc = np.full(num_repeats, np.nan)
time_ata_smc = np.full(num_repeats, np.nan)

log_evidence_ta_smc = np.full(num_repeats, np.nan)
count_ta_smc = np.full(num_repeats, np.nan)
time_ta_smc = np.full(num_repeats, np.nan)

log_evidence_ans_smc = np.full(num_repeats, np.nan)
count_ans_smc = np.full(num_repeats, np.nan)
time_ans_smc = np.full(num_repeats, np.nan)

log_evidence_ns_smc = np.full(num_repeats, np.nan)
count_ns_smc = np.full(num_repeats, np.nan)
time_ns_smc = np.full(num_repeats, np.nan)
error_flag_ns_smc = np.full(num_repeats, np.nan)

log_evidence_ns = np.full(num_repeats, np.nan)
count_ns = np.full(num_repeats, np.nan)
time_ns = np.full(num_repeats, np.nan)

log_evidence_ns_star = np.full(num_repeats, np.nan)
count_ns_star = np.full(num_repeats, np.nan)
time_ns_star = np.full(num_repeats, np.nan)

for i in range(num_repeats):
    
    # %%%%%%%%%%%%%%%%%%%% PERFORMING ATA-SMC %%%%%%%%%%%%%%%%%%%%%%% 
    options = Options()
    options.N = N
    options.d = d
    options.alpha = 0.999
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    # log_unnormalized_w, normalized_w between time t and t+1
    log_evidence, count_loglike, gammavar, ess, log_unnormalized_w, normalized_w, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, log_evidence_t, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = ATA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ata_smc[i] = log_evidence
    count_ata_smc[i] = count_loglike
    time_ata_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED TA-SMC %%%%%%%%%%%%%%%%%%%%%%%
    options.N = N
    options.d = d
    options.gammavar = gammavar
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i + num_repeats)
    t_start = time.time()
    log_evidence, count_loglike = TA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ta_smc[i] = log_evidence
    count_ta_smc[i] = count_loglike + count_ata_smc[i]
    time_ta_smc[i] = t_elapsed + time_ata_smc[i]

    # %%%%%%%%%%%%%%%%%%%%%% PERFORMING ANS-SMC %%%%%%%%%%%%%%%%%%%%% 
    options = Options()
    options.N = N
    options.d = d
    options.alpha = alpha
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    # Log unnormalized and normalized weights for the eliminated particles at each level t
    llog_evidence, count_loglike, levels, log_unnormalized_w, log_normalized_w, log_evidence_t, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = ANS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ans_smc[i] = log_evidence
    count_ans_smc[i] = count_loglike
    time_ans_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED NS-SMC %%%%%%%%%%%%%%%%%%%%%% -1
    options.N = N
    options.d = d
    options.levels = levels # levels is returned from ANS-SMC
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i + num_repeats)
    t_start = time.time()
    # Log unnormalized and normalized weights for the eliminated particles at each level t
    log_evidence, count_loglike, error_flag, log_unnormalized_w, log_normalized_w, log_evidence_t, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = NS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ns_smc[i] = log_evidence
    count_ns_smc[i] = count_loglike + count_ans_smc[i]
    time_ns_smc[i] = t_elapsed + time_ans_smc[i]
    error_flag_ns_smc[i] = error_flag

    # %%%%%%%%%%%%%%%%% PERFORMING NS (WITH MAX EVALS) %%%%%%%%%%%%%%%%%%%%%%
    options_ns = Options()
    options_ns.N = N
    options_ns.d = d
    options_ns.desired_count = count_ns_smc[i]
    options_ns.R = repeats
    options_ns.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    log_evidence, log_evidence_star_val, count_loglike = NS_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options_ns, False)
    t_elapsed = time.time() - t_start

    log_evidence_ns[i] = log_evidence
    count_ns[i] = count_loglike
    time_ns[i] = t_elapsed
    
    log_evidence_ns_star[i] = log_evidence_star_val
    count_ns_star[i] = count_loglike
    time_ns_star[i] = t_elapsed

    # Print progress update -1
    if (i + 1) % max(1, num_repeats // 100) == 0:
        print(f"Iteration {i + 1}.")

Iteration 10.
Iteration 20.
Iteration 30.
Iteration 40.
Iteration 50.
Iteration 60.
Iteration 70.
Iteration 80.
Iteration 90.
Iteration 100.
Iteration 110.
Iteration 120.
Iteration 130.
Iteration 140.
Iteration 150.
Iteration 160.
Iteration 170.
Iteration 180.
Iteration 190.
Iteration 200.
Iteration 210.
Iteration 220.
Iteration 230.
Iteration 240.
Iteration 250.
Iteration 260.
Iteration 270.
Iteration 280.
Iteration 290.
Iteration 300.
Iteration 310.
Iteration 320.
Iteration 330.
Iteration 340.
Iteration 350.
Iteration 360.
Iteration 370.
Iteration 380.
Iteration 390.
Iteration 400.
Iteration 410.
Iteration 420.
Iteration 430.
Iteration 440.
Iteration 450.
Iteration 460.
Iteration 470.
Iteration 480.
Iteration 490.
Iteration 500.
Iteration 510.
Iteration 520.
Iteration 530.
Iteration 540.
Iteration 550.
Iteration 560.
Iteration 570.
Iteration 580.
Iteration 590.
Iteration 600.
Iteration 610.
Iteration 620.
Iteration 630.
Iteration 640.
Iteration 650.
Iteration 660.
Iteration 670.
Iter

In [22]:
d = 10
u, v = 0.1, 0.01
a1, a2 = 0.1, 0.9
Z_true = a1 * gammainc(d / 2.0, 1.0 / (2.0 * u**2)) + a2 * gammainc(d / 2.0, 1.0 / (2.0 * v**2))
Z_true_paper = 0.39210 

methods = [
    ("NS", log_evidence_ns, count_ns),
    ("NS*", log_evidence_ns_star, count_ns_star),
    ("ANS-SMC", log_evidence_ans_smc, count_ans_smc),
    ("NS-SMC", log_evidence_ns_smc, count_ns_smc),
    ("ATA-SMC", log_evidence_ata_smc, count_ata_smc),
    ("TA-SMC", log_evidence_ta_smc, count_ta_smc),
]

print(f"--- Table 2 Results (MCMC Sampler, N = {N}, {num_repeats} repeats) ---")
print(f"{'Method':<10} | {'Z^N (SE)':<18} | {'Evals':<10} | {'MSE':<12} | {'p-val (<0.05/30?)'}")
print("-" * 75)

mse_list = []
results_summary = []

for name, log_ev_arr, count_arr in methods:
    # Convert from log-evidence to raw evidence Z
    Z_est = np.exp(log_ev_arr)
    
    mean_Z = np.mean(Z_est)
    se_Z = np.std(Z_est, ddof=1) / np.sqrt(len(Z_est))
    mean_evals = np.mean(count_arr)
    
    # MSE and t-test against ground truth
    mse = np.mean((Z_est - Z_true_paper)**2)
    t_stat, p_val = stats.ttest_1samp(Z_est, Z_true_paper)
    is_biased = p_val < (0.05 / 30.0)
    
    mse_list.append(mse)
    results_summary.append((name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased))

best_mse_idx = np.argmin(mse_list)

for idx, (name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased) in enumerate(results_summary):
    flag_italic = "ITALIC (Biased)" if is_biased else ""
    flag_bold = "BOLD (Best MSE)" if idx == best_mse_idx else ""
    flags = " ".join(filter(None, [flag_bold, flag_italic]))
    
    print(f"{name:<10} | {mean_Z:.4f} ({se_Z:.4f})   | {mean_evals:.1e}   | {mse:.6e} | {p_val:.4f} {flags}")

--- Table 2 Results (MCMC Sampler, N = 1000, 1000 repeats) ---
Method     | Z^N (SE)           | Evals      | MSE          | p-val (<0.05/30?)
---------------------------------------------------------------------------
NS         | 0.4052 (0.0068)   | 9.9e+05   | 4.641547e-02 | 0.0541 
NS*        | 0.3989 (0.0067)   | 9.9e+05   | 4.488430e-02 | 0.3137 
ANS-SMC    | 0.0530 (0.0115)   | 5.0e+05   | 2.482584e-01 | 0.0000 ITALIC (Biased)
NS-SMC     | 0.4048 (0.0051)   | 9.9e+05   | 2.607475e-02 | 0.0129 BOLD (Best MSE)
ATA-SMC    | 0.1274 (0.0428)   | 1.7e+06   | 1.898045e+00 | 0.0000 ITALIC (Biased)
TA-SMC     | 0.0530 (0.0115)   | 3.4e+06   | 2.482584e-01 | 0.0000 ITALIC (Biased)


In [ ]:
# N = 10000
class Options:
    pass

N = 10000 # Run for 100, 1000, and 10000
d = 10
alpha = np.exp(-1)
repeats = 10

if N == 100:
    num_repeats = 10000
elif N == 1000:
    num_repeats = 1000
elif N == 10000:
    num_repeats = 100

# Initialize result arrays with NaNs
log_evidence_ata_smc = np.full(num_repeats, np.nan)
count_ata_smc = np.full(num_repeats, np.nan)
time_ata_smc = np.full(num_repeats, np.nan)

log_evidence_ta_smc = np.full(num_repeats, np.nan)
count_ta_smc = np.full(num_repeats, np.nan)
time_ta_smc = np.full(num_repeats, np.nan)

log_evidence_ans_smc = np.full(num_repeats, np.nan)
count_ans_smc = np.full(num_repeats, np.nan)
time_ans_smc = np.full(num_repeats, np.nan)

log_evidence_ns_smc = np.full(num_repeats, np.nan)
count_ns_smc = np.full(num_repeats, np.nan)
time_ns_smc = np.full(num_repeats, np.nan)
error_flag_ns_smc = np.full(num_repeats, np.nan)

log_evidence_ns = np.full(num_repeats, np.nan)
count_ns = np.full(num_repeats, np.nan)
time_ns = np.full(num_repeats, np.nan)

log_evidence_ns_star = np.full(num_repeats, np.nan)
count_ns_star = np.full(num_repeats, np.nan)
time_ns_star = np.full(num_repeats, np.nan)

for i in range(num_repeats):
    
    # %%%%%%%%%%%%%%%%%%%% PERFORMING ATA-SMC %%%%%%%%%%%%%%%%%%%%%%% 
    options = Options()
    options.N = N
    options.d = d
    options.alpha = 0.999
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    # log_unnormalized_w, normalized_w between time t and t+1
    log_evidence, count_loglike, gammavar, ess, log_unnormalized_w, normalized_w, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, log_evidence_t, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = ATA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ata_smc[i] = log_evidence
    count_ata_smc[i] = count_loglike
    time_ata_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED TA-SMC %%%%%%%%%%%%%%%%%%%%%%%
    options.N = N
    options.d = d
    options.gammavar = gammavar
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i + num_repeats)
    t_start = time.time()
    log_evidence, count_loglike = TA_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ta_smc[i] = log_evidence
    count_ta_smc[i] = count_loglike + count_ata_smc[i]
    time_ta_smc[i] = t_elapsed + time_ata_smc[i]

    # %%%%%%%%%%%%%%%%%%%%%% PERFORMING ANS-SMC %%%%%%%%%%%%%%%%%%%%% 
    options = Options()
    options.N = N
    options.d = d
    options.alpha = alpha
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    # Log unnormalized and normalized weights for the eliminated particles at each level t
    llog_evidence, count_loglike, levels, log_unnormalized_w, log_normalized_w, log_evidence_t, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = ANS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start

    log_evidence_ans_smc[i] = log_evidence
    count_ans_smc[i] = count_loglike
    time_ans_smc[i] = t_elapsed

    # %%%%%%%%%%%%%%%%%%%%%%%% PERFORMING FIXED NS-SMC %%%%%%%%%%%%%%%%%%%%%% 
    options.N = N
    options.d = d
    options.levels = levels # levels is returned from ANS-SMC
    options.R = repeats
    options.sig = 0.1

    np.random.seed(i + num_repeats)
    t_start = time.time()
    # Log unnormalized and normalized weights for the eliminated particles at each level t
    log_evidence, count_loglike, error_flag, log_unnormalized_w, log_normalized_w, log_evidence_t, prop_particles_inside_spike_radius_1sd, prop_particles_inside_spike_radius_2sd, prop_particles_inside_spike_radius_3sd, accept_rate_t, accept_spike_t, accept_slab_t, total_prop_spike, total_prop_slab = NS_SMC_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options, False)
    t_elapsed = time.time() - t_start
    
    log_evidence_ns_smc[i] = log_evidence
    count_ns_smc[i] = count_loglike + count_ans_smc[i]
    time_ns_smc[i] = t_elapsed + time_ans_smc[i]
    error_flag_ns_smc[i] = error_flag

    # %%%%%%%%%%%%%%%%% PERFORMING NS (WITH MAX EVALS) %%%%%%%%%%%%%%%%%%%%%%
    options_ns = Options()
    options_ns.N = N
    options_ns.d = d
    options_ns.desired_count = count_ns_smc[i]
    options_ns.R = repeats
    options_ns.sig = 0.1

    np.random.seed(i)
    t_start = time.time()
    log_evidence, log_evidence_star_val, count_loglike = NS_bespoke_RW(loglike_fn, logprior_fn, simprior_fn, options_ns, False)
    t_elapsed = time.time() - t_start

    log_evidence_ns[i] = log_evidence
    count_ns[i] = count_loglike
    time_ns[i] = t_elapsed
    
    log_evidence_ns_star[i] = log_evidence_star_val
    count_ns_star[i] = count_loglike
    time_ns_star[i] = t_elapsed

    # Print progress update -1
    if (i + 1) % max(1, num_repeats // 100) == 0:
        print(f"Iteration {i + 1}.")

Iteration 1.
Iteration 2.
Iteration 3.
Iteration 4.
Iteration 5.
Iteration 6.
Iteration 7.
Iteration 8.
Iteration 9.
Iteration 10.
Iteration 11.
Iteration 12.
Iteration 13.
Iteration 14.
Iteration 15.
Iteration 16.
Iteration 17.
Iteration 18.
Iteration 19.
Iteration 20.
Iteration 21.
Iteration 22.
Iteration 23.
Iteration 24.
Iteration 25.
Iteration 26.
Iteration 27.
Iteration 28.
Iteration 29.
Iteration 30.
Iteration 31.
Iteration 32.
Iteration 33.
Iteration 34.
Iteration 35.
Iteration 36.
Iteration 37.
Iteration 38.
Iteration 39.
Iteration 40.
Iteration 41.
Iteration 42.
Iteration 43.
Iteration 44.
Iteration 45.
Iteration 46.
Iteration 47.
Iteration 48.
Iteration 49.
Iteration 50.
Iteration 51.
Iteration 52.
Iteration 53.
Iteration 54.
Iteration 55.
Iteration 56.
Iteration 57.
Iteration 58.
Iteration 59.
Iteration 60.
Iteration 61.
Iteration 62.
Iteration 63.
Iteration 64.
Iteration 65.
Iteration 66.
Iteration 67.
Iteration 68.
Iteration 69.
Iteration 70.
Iteration 71.
Iteration 72.
I

In [25]:
d = 10
u, v = 0.1, 0.01
a1, a2 = 0.1, 0.9
Z_true = a1 * gammainc(d / 2.0, 1.0 / (2.0 * u**2)) + a2 * gammainc(d / 2.0, 1.0 / (2.0 * v**2))
Z_true_paper = 0.39210 

methods = [
    ("NS", log_evidence_ns, count_ns),
    ("NS*", log_evidence_ns_star, count_ns_star),
    ("ANS-SMC", log_evidence_ans_smc, count_ans_smc),
    ("NS-SMC", log_evidence_ns_smc, count_ns_smc),
    ("ATA-SMC", log_evidence_ata_smc, count_ata_smc),
    ("TA-SMC", log_evidence_ta_smc, count_ta_smc),
]

print(f"--- Table 2 Results (MCMC Sampler, N = {N}, {num_repeats} repeats) ---")
print(f"{'Method':<10} | {'Z^N (SE)':<18} | {'Evals':<10} | {'MSE':<12} | {'p-val (<0.05/30?)'}")
print("-" * 75)

mse_list = []
results_summary = []

for name, log_ev_arr, count_arr in methods:
    # Convert from log-evidence to raw evidence Z
    Z_est = np.exp(log_ev_arr)
    
    mean_Z = np.mean(Z_est)
    se_Z = np.std(Z_est, ddof=1) / np.sqrt(len(Z_est))
    mean_evals = np.mean(count_arr)
    
    # MSE and t-test against ground truth
    mse = np.mean((Z_est - Z_true_paper)**2)
    t_stat, p_val = stats.ttest_1samp(Z_est, Z_true_paper)
    is_biased = p_val < (0.05 / 30.0)
    
    mse_list.append(mse)
    results_summary.append((name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased))

best_mse_idx = np.argmin(mse_list)

for idx, (name, mean_Z, se_Z, mean_evals, mse, p_val, is_biased) in enumerate(results_summary):
    flag_italic = "ITALIC (Biased)" if is_biased else ""
    flag_bold = "BOLD (Best MSE)" if idx == best_mse_idx else ""
    flags = " ".join(filter(None, [flag_bold, flag_italic]))
    
    print(f"{name:<10} | {mean_Z:.4f} ({se_Z:.4f})   | {mean_evals:.1e}   | {mse:.6e} | {p_val:.4f} {flags}")

--- Table 2 Results (MCMC Sampler, N = 10000, 100 repeats) ---
Method     | Z^N (SE)           | Evals      | MSE          | p-val (<0.05/30?)
---------------------------------------------------------------------------
NS         | 0.3978 (0.0057)   | 9.8e+06   | 3.253795e-03 | 0.3240 
NS*        | 0.3971 (0.0057)   | 9.8e+06   | 3.237035e-03 | 0.3798 
ANS-SMC    | 3.8224 (3.7734)   | 4.9e+06   | 1.421374e+03 | 0.3655 
NS-SMC     | 0.3958 (0.0048)   | 9.8e+06   | 2.307767e-03 | 0.4397 BOLD (Best MSE)
ATA-SMC    | 0.8258 (0.5258)   | 1.8e+07   | 2.756074e+01 | 0.4115 
TA-SMC     | 3.8224 (3.7734)   | 3.6e+07   | 1.421374e+03 | 0.3655 
